# HEEDB: MUSE vs physician overread differences

How much the physician overreads (the ground truth) differ from the MUSE statements they correct, on the HEEDB by-label splits with blank overreads dropped (the same data as the HEEDB table in `results/data-tables.ipynb`).

Labels come from `label_src`: `audited_physician` (default) for the overread and `audited_muse` for MUSE, or `original_physician` and `original_muse`. The audited sources only count codes in the main read, not the comparison text (see `protossl/datasets/_heedb_label_rules.py`). Some `audited_physician` (ECG, label) entries are unknown (`UNKNOWN_LABEL`); `audited_muse` has no unknowns. For each label:

- **Physician = 1**: ECGs positive in the physician overread, as a % of the split.
- **Physician unknown** (audited only): ECGs whose overread label is unknown, as a % of the split.
- **MUSE = 1**: ECGs MUSE called positive, as a % of the split.
- **Removed by physician**: MUSE positive, physician negative, as a % of MUSE positives. With unknowns, only MUSE positives with a known overread label count in the denominator.
- **Added by physician**: MUSE negative, physician positive, as a % of physician positives.

Labels with few removals and additions have ground truth that is close to MUSE's own output, so MUSE's measured accuracy on those labels is favored by construction.

In [ ]:
import numpy as np
import pandas as pd

from protossl.datasets import HeedbECGDataset
from protossl.defines import HEEDB_TARGETS, UNKNOWN_LABEL

dataset_path = "/opt/gpudata/ecg/heedb"
splits = ["train", "val", "test"]
data_kwargs = {
    "heedb_split_type": "by-label",
    "drop_blank_overread": True,
}
# ground truth labels: "original_physician" or "audited_physician" (has unknowns)
label_src = "audited_physician"
# MUSE labels read the same way as the ground truth: audited_muse only counts main-read
# codes, original_muse also counts the comparison text (which often describes a prior ECG)
muse_label_src = "original_muse" if label_src == "original_physician" else "audited_muse"
# original_physician keeps the original file name
csv_suffix = "" if label_src == "original_physician" else f"-{label_src}"
output_csv = f"temp_heedb_by_label_drop_blank_muse_overread{csv_suffix}.csv"

In [ ]:
phys_labels = dict()
muse_labels = dict()
for split in splits:
    ds_phys = HeedbECGDataset(
        dataset_path=dataset_path,
        split=split,
        sampling_rate=100,
        label_src=label_src,
        **data_kwargs,
    )
    ds_muse = HeedbECGDataset(
        dataset_path=dataset_path,
        split=split,
        sampling_rate=100,
        label_src=muse_label_src,
        **data_kwargs,
    )
    assert (ds_phys.sample_ids == ds_muse.sample_ids).all()
    # 1 / 0 / UNKNOWN_LABEL, so not cast to bool
    phys_labels[split] = ds_phys.labels.numpy()
    muse_labels[split] = ds_muse.labels.numpy()
    assert (muse_labels[split] != UNKNOWN_LABEL).all()

has_unknown = any((phys_labels[s] == UNKNOWN_LABEL).any() for s in splits)

In [ ]:
def fmt(count, denom):
    # same "n (%)" format as TableOne in results/data-tables.ipynb
    if denom == 0:
        # e.g. audited_muse has no VT positives in val
        return f"{count} (-)"
    return f"{count} ({100.0 * count / denom:.1f})"


for s in splits:
    known = phys_labels[s] != UNKNOWN_LABEL
    n_same = int(((phys_labels[s] == muse_labels[s]) | ~known).all(axis=1).sum())
    on = "all known labels" if has_unknown else "all labels"
    print(f"{s}: overread identical to MUSE on {on} for {fmt(n_same, len(phys_labels[s]))} of {len(phys_labels[s])} ECGs")

removed_denom = "MUSE = 1 with known overread" if has_unknown else "MUSE = 1"
index, rows = [], []
for i, label in enumerate(HEEDB_TARGETS):
    p = {s: phys_labels[s][:, i] == 1 for s in splits}
    u = {s: phys_labels[s][:, i] == UNKNOWN_LABEL for s in splits}
    m = {s: muse_labels[s][:, i] == 1 for s in splits}
    # title case to match the HEEDB table in results/data-tables.ipynb
    label = label.title()
    index.append((label, "Physician = 1, n (%)"))
    rows.append([fmt(int(p[s].sum()), len(p[s])) for s in splits])
    if has_unknown:
        index.append((label, "Physician unknown, n (%)"))
        rows.append([fmt(int(u[s].sum()), len(u[s])) for s in splits])
    index += [
        (label, "MUSE = 1, n (%)"),
        (label, f"Removed by physician, n (% of {removed_denom})"),
        (label, "Added by physician, n (% of physician = 1)"),
    ]
    rows += [
        [fmt(int(m[s].sum()), len(m[s])) for s in splits],
        # physician negative means known and not positive, so unknowns are neither removed nor added
        [fmt(int((m[s] & ~p[s] & ~u[s]).sum()), int((m[s] & ~u[s]).sum())) for s in splits],
        [fmt(int((~m[s] & p[s]).sum()), int(p[s].sum())) for s in splits],
    ]
rows_per_label = len(index) // len(HEEDB_TARGETS)

table = pd.DataFrame(
    rows,
    index=pd.MultiIndex.from_tuples(index, names=["Label", ""]),
    columns=pd.MultiIndex.from_tuples([("Split", s) for s in splits]),
)

In [ ]:
table.to_csv(output_csv)
print(table.to_latex(multirow=True))

In [ ]:
import pandas as pd
from IPython.display import display

def column_widths(df, px_per_char=8, padding=16, max_px=1000):
    """Pixel width per column, based on the longest value in the FULL df."""
    return {
        col: min(max(len(str(col)), df[col].astype(str).str.len().max()) * px_per_char + padding, max_px)
        for col in df.columns
    }

def index_widths(df, label_px=150, px_per_char=6, padding=16):
    """Pixel width per index level: fixed for the label (wraps), longest value for the measure (no wrap)."""
    measure_px = max(len(str(v)) for v in df.index.get_level_values(1)) * px_per_char + padding
    return label_px, measure_px

def styled_chunk(chunk, widths, index_px):
    label_px, measure_px = index_px
    total = label_px + measure_px + sum(widths.values())
    styles = [
        {'selector': '', 'props': f'table-layout: fixed; width: {total}px;'},
        {'selector': 'th, td', 'props': 'overflow: hidden; text-overflow: ellipsis; white-space: nowrap;'},
        # label column: header cells are the first child of each header row
        {'selector': 'thead th:nth-child(1), th.row_heading.level0', 'props': f'width: {label_px}px; white-space: normal; overflow-wrap: break-word; vertical-align: top;'},
        # measure column
        {'selector': 'thead th:nth-child(2), th.row_heading.level1', 'props': f'width: {measure_px}px;'},
        # {'selector': 'th, td', 'props': 'white-space: normal; overflow-wrap: anywhere; vertical-align: top;'},
    ] + [
        {'selector': f'th.col_heading.col{i}', 'props': f'width: {widths[col]}px;'}
        for i, col in enumerate(chunk.columns)
    ]
    # shade every other label group and rule a line between groups, instead of the default per-row striping
    labels = chunk.index.get_level_values(0)
    shaded = pd.factorize(labels)[0] % 2 == 1
    first = labels != np.roll(labels, 1)
    row_css = [
        ("background-color: rgba(127, 127, 127, 0.15);" if s else "")
        + ("border-top: 1px solid rgba(127, 127, 127, 0.5);" if f else "")
        for s, f in zip(shaded, first)
    ]
    styles.append({'selector': 'tbody tr', 'props': 'background-color: transparent !important;'})
    # fixed layout takes column widths from the first header row, so drop the "Split" level that spans all columns
    return (
        chunk.droplevel(0, axis=1).style
        .set_table_styles(styles)
        .apply(lambda col: row_css, axis=0)
        .apply_index(lambda idx: row_css, axis=0, level=[0, 1])
    )

widths = column_widths(table)   # computed once, outside the loop
index_px = index_widths(table)

In [ ]:
# 5 labels per chunk, so a label's rows never split across chunks
chunk_rows = 5 * rows_per_label
with pd.option_context("display.max_rows", None):
    for start in range(0, len(table), chunk_rows):
        display(styled_chunk(table.iloc[start:start + chunk_rows], widths, index_px))

In [ ]:
# PNG/PDF of the full table for sharing, next to output_csv
from _table_figure import save_table

unknown_note = (
    " Physician unknown: the overread can't be called either way for that label."
    if has_unknown else ""
)
save_table(
    table,
    output_csv,
    title=f"HEEDB: MUSE ({muse_label_src}) vs physician overread ({label_src})",
    note=(
        "By-label splits, blank overreads dropped. Removed by physician: MUSE positive, "
        f"physician negative. Added by physician: MUSE negative, physician positive.{unknown_note}"
    ),
)